# 03 · Análise de erros do adaptador publicado

Carrega o adaptador do Hugging Face, prevê o **conjunto de teste (300 avaliações revisadas)** e organiza os erros para você escrever a análise no README:

1. confere se as métricas batem com as do notebook 02 (sanidade);
2. relatório por classe e matrizes de confusão por aspecto;
3. classifica **cada erro automaticamente** (menção perdida, menção falsa, polaridade invertida, neutro confundido) e conta por aspecto;
4. mostra erros por nota e por tamanho do texto;
5. exporta `results/erros_modelo.csv` para você categorizar à mão.

**Execução:** Colab com **GPU T4**. Envie o `data.zip` (precisa só do `test.jsonl`).

In [ ]:
!pip -q install peft bitsandbytes accelerate

In [ ]:
import os, json, re, zipfile, numpy as np, pandas as pd, torch
assert torch.cuda.is_available(), "Sem GPU: Ambiente de execução > Alterar tipo de ambiente de execução > T4 GPU."

from google.colab import files
up = files.upload()                                   # selecione o data.zip
zipfile.ZipFile(next(iter(up))).extractall(".")
!ls -la data

Saving data.zip to data.zip
total 592
drwxr-xr-x 2 root root   4096 Nov 10 21:29 .
drwxr-xr-x 1 root root   4096 Nov 10 21:29 ..
-rw-r--r-- 1 root root     42 Nov 10 21:29 labeler.txt
-rw-r--r-- 1 root root  59659 Nov 10 21:29 test.jsonl
-rw-r--r-- 1 root root 468273 Nov 10 21:29 train.jsonl
-rw-r--r-- 1 root root  60063 Nov 10 21:29 val.jsonl


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

HF_USER = "thiag8ito"                 # seu usuário no Hugging Face
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
REPO = f"{HF_USER}/qwen2.5-1.5b-aspectos-ecommerce-ptbr-qlora"
ASPECTS = ["produto", "entrega", "atendimento"]
LABELS = ["positivo", "negativo", "neutro", "ausente"]

tok = AutoTokenizer.from_pretrained(MODEL_ID); tok.padding_side = "left"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map={"": 0}, dtype=torch.float16)
model = PeftModel.from_pretrained(base, REPO).eval()      # repositório público; se for privado, faça login antes

test = pd.read_json("data/test.jsonl", lines=True)
print(len(test), "avaliações de teste")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

... (3 outputs left out)

adapter_config.json:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 73.9MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

300 avaliações de teste


In [ ]:
SYSTEM = """Você classifica avaliações de clientes de e-commerce brasileiro por aspecto.
Aspectos: produto (qualidade/estado do item), entrega (prazo/transporte), atendimento (vendedor/suporte/troca).
Valores: positivo, negativo, neutro, ausente (aspecto não mencionado).
Responda somente com JSON: {"produto":"...","entrega":"...","atendimento":"..."}"""    # idêntico ao do treino

def build_messages(text):
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}]

@torch.no_grad()
def predict(m, texts, bs=16, max_new_tokens=40):
    m.config.use_cache = True
    order = np.argsort([len(t) for t in texts]); outs = [None] * len(texts)
    for i in range(0, len(texts), bs):
        idx = order[i:i + bs]
        prompts = [tok.apply_chat_template(build_messages(texts[j]), tokenize=False, add_generation_prompt=True) for j in idx]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(m.device)
        gen = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, temperature=None,
                         top_p=None, top_k=None, repetition_penalty=1.0, pad_token_id=tok.pad_token_id)
        dec = tok.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
        for j, d in zip(idx, dec): outs[j] = d
    return outs

def parse(out):
    mt = re.search(r"\{.*?\}", out, re.S)
    if not mt: return None
    try: d = json.loads(mt.group(0))
    except Exception: return None
    if set(d) != set(ASPECTS) or any(d[a] not in LABELS for a in ASPECTS): return None
    return d

## 1) Previsões e checagem de sanidade

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, classification_report

raw = predict(model, test.text.tolist())
parsed = [parse(o) for o in raw]
for a in ASPECTS: test[f"pred_{a}"] = [(p[a] if p else "invalido") for p in parsed]

f1s = []
for a in ASPECTS:
    labs = [l for l in LABELS if l in set(test[a])]
    f1 = f1_score(test[a], test[f"pred_{a}"], labels=labs, average="macro", zero_division=0); f1s.append(f1)
    print(f"{a:12s} F1 macro={f1:.3f}  acc={accuracy_score(test[a], test[f'pred_{a}']):.3f}")
exact = np.mean(np.all([test[a] == test[f"pred_{a}"] for a in ASPECTS], axis=0))
print(f"\nF1 macro médio={np.mean(f1s):.3f} (notebook 02: 0.691) | exact match={exact:.3f} (0.733) | JSON válido={np.mean([p is not None for p in parsed]):.2f} (1.00)")
print("Diferenças pequenas são normais (quantização e padding em lote); diferenças grandes indicam outro teste ou outro prompt.")

produto      F1 macro=0.805  acc=0.877
entrega      F1 macro=0.690  acc=0.913
atendimento  F1 macro=0.577  acc=0.887

F1 macro médio=0.691 (notebook 02: 0.691) | exact match=0.733 (0.733) | JSON válido=1.00 (1.00)
Diferenças pequenas são normais (quantização e padding em lote); diferenças grandes indicam outro teste ou outro prompt.


## 2) Relatório por classe e matrizes de confusão

In [ ]:
for a in ASPECTS:
    print(f"\n=== {a} ===")
    print(classification_report(test[a], test[f"pred_{a}"], zero_division=0, digits=3))
    display(pd.crosstab(test[a], test[f"pred_{a}"], rownames=["gold (humano)"], colnames=["previsto"]))


=== produto ===
              precision    recall  f1-score   support

     ausente      0.808     0.881     0.843        67
    negativo      0.903     0.886     0.894       105
      neutro      0.611     0.478     0.537        23
    positivo      0.943     0.952     0.948       105

    accuracy                          0.877       300
   macro avg      0.816     0.799     0.805       300
weighted avg      0.874     0.877     0.874       300



previsto,ausente,negativo,neutro,positivo
gold (humano),,,,
ausente,59,5,1,2
negativo,7,93,3,2
neutro,6,4,11,2
positivo,1,1,3,100



=== entrega ===
              precision    recall  f1-score   support

     ausente      0.921     0.955     0.938       134
    negativo      0.898     0.908     0.903        87
      neutro      0.000     0.000     0.000         2
    positivo      0.971     0.870     0.918        77

    accuracy                          0.913       300
   macro avg      0.697     0.683     0.690       300
weighted avg      0.921     0.913     0.916       300



previsto,ausente,negativo,neutro,positivo
gold (humano),,,,
ausente,128,6,0,0
negativo,4,79,2,2
neutro,1,1,0,0
positivo,6,2,2,67



=== atendimento ===
              precision    recall  f1-score   support

     ausente      0.906     0.975     0.939       238
    negativo      0.839     0.684     0.754        38
      neutro      0.000     0.000     0.000        10
    positivo      0.667     0.571     0.615        14

    accuracy                          0.887       300
   macro avg      0.603     0.558     0.577       300
weighted avg      0.856     0.887     0.869       300



previsto,ausente,negativo,neutro,positivo
gold (humano),,,,
ausente,232,3,1,2
negativo,11,26,0,1
neutro,7,2,0,1
positivo,6,0,0,8


## 3) Erros classificados automaticamente

Tipos (por aspecto):
- **menção perdida**: o humano marcou o aspecto como presente e o modelo previu `ausente`;
- **menção falsa**: o humano marcou `ausente` e o modelo previu algum sentimento;
- **polaridade invertida**: `positivo` ↔ `negativo`;
- **neutro confundido**: `neutro` trocado por `positivo`/`negativo`.

In [ ]:
def tipo(g, p):
    if p == "invalido": return "saída inválida"
    if g == "ausente": return "menção falsa (previu o que não existe)"
    if p == "ausente": return "menção perdida (ignorou aspecto presente)"
    if {g, p} == {"positivo", "negativo"}: return "polaridade invertida"
    return "neutro confundido com polaridade"

rows = []
for _, r in test.iterrows():
    for a in ASPECTS:
        if r[a] != r[f"pred_{a}"]:
            rows.append({"id": r["id"], "aspecto": a, "gold": r[a], "previsto": r[f"pred_{a}"],
                         "tipo_erro": tipo(r[a], r[f"pred_{a}"]), "nota": r["review_score"], "texto": r["text"]})
err = pd.DataFrame(rows)
print(f"{len(err)} erros (aspecto x avaliação) em {err.id.nunique()} de {len(test)} avaliações\n")
display(pd.crosstab(err["aspecto"], err["tipo_erro"], margins=True))
print("\nTrocas mais frequentes (gold -> previsto):")
display(err.groupby(["gold", "previsto"]).size().sort_values(ascending=False).head(10).to_frame("n"))

for t in err.tipo_erro.value_counts().index:
    print(f"\n### {t}")
    for _, r in err[err.tipo_erro == t].head(5).iterrows():
        print(f"- [{r.aspecto}] gold={r.gold} previsto={r.previsto} | {r.texto[:160]}")

97 erros (aspecto x avaliação) em 80 de 300 avaliações



tipo_erro,menção falsa (previu o que não existe),menção perdida (ignorou aspecto presente),neutro confundido com polaridade,polaridade invertida,All
aspecto,,,,,
atendimento,6,24,3,1,34
entrega,6,11,5,4,26
produto,8,14,12,3,37
All,20,49,20,8,97



Trocas mais frequentes (gold -> previsto):


n
gold     previsto    
negativo ausente   22
ausente  negativo  14
neutro   ausente   14
positivo ausente   13
neutro   negativo   7
positivo neutro     5
negativo neutro     5
         positivo   5
ausente  positivo   4
neutro   positivo   3


### menção perdida (ignorou aspecto presente)
- [atendimento] gold=positivo previsto=ausente | Não Recomendo. O produto veio na caixa original, sem qualquer proteção, o correio durante a entrega rasgou toda a caixa, e ao testar, constatei que somente um d
- [atendimento] gold=neutro previsto=ausente | Bem gostei mais as bonecas so tem cabelo na frente e so nao troquei por que nao consegui.era presente e fiquei na mão
- [produto] gold=negativo previsto=ausente | não recomendo porque até o momento não recebi meu produto,nem tive nenhuma informação da empresa responsavél
- [atendimento] gold=negativo previsto=ausente | Falta um.produto. Não recebi a capa De sofá. Estou aguardando e não estou informada de quando virá
- [atendimento] gold=positivo previsto=ausente | Gostei muito do compromisso e pontualidade

### menção falsa (previu o que não existe)
- [produto] gold=ausente previsto=positivo | Estou usando ainda. Chegou hoje.so tem como avaliar a entrega
- [produto] gold=ausente previsto

## 4) Onde o modelo erra mais: nota e tamanho do texto

In [ ]:
test["n_erros"] = sum((test[a] != test[f"pred_{a}"]).astype(int) for a in ASPECTS)
test["faixa_tamanho"] = pd.cut(test.text.str.len(), [0, 60, 120, 250, 400], labels=["até 60", "61-120", "121-250", "251-400"])
print("Erros por avaliação, por nota:");    display(test.groupby("review_score").n_erros.mean().round(2).to_frame("media_de_erros"))
print("Erros por avaliação, por tamanho:"); display(test.groupby("faixa_tamanho", observed=True).n_erros.mean().round(2).to_frame("media_de_erros"))

Erros por avaliação, por nota:


,media_de_erros
review_score,
1,0.48
2,0.31
3,0.31
4,0.30
5,0.21


Erros por avaliação, por tamanho:


,media_de_erros
faixa_tamanho,
até 60,0.20
61-120,0.37
121-250,0.42


## 5) Exportar para categorizar à mão

O CSV traz uma linha por erro, com colunas vazias `categoria_manual` e `comentario`. Abra no Sheets, leia pelo menos **30 a 50 erros** e preencha com categorias como: aspecto implícito, ironia, avaliação mista, texto muito curto, **rótulo humano discutível**, outros. Conte quantos erros cabem em cada categoria e use isso no README.

In [ ]:
err["categoria_manual"] = ""; err["comentario"] = ""
os.makedirs("results", exist_ok=True)
err.to_csv("results/erros_modelo.csv", index=False)
files.download("results/erros_modelo.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>